# 第68课 · 100 帧如何吐出 5 个字——CTC 对齐：blank 符号与标签折叠（label collapse）

**目标**：理解 CTC 的核心思想——blank 符号 + 去重规则 + 路径求和，让模型在无帧级对齐标注的情况下完成序列训练。

🔗 **Aurora 连接**：`aurora.speech` 将调用 `torch.nn.CTCLoss`；读懂 CTC 是理解 Whisper 训练目标的前提。 本节手写贪婪解码，建立对 CTC 路径压缩的直觉。

← **上一课**　[L67 · Edit Distance 从零实现](L67_edit_distance.ipynb)

> 上节课学习了 **Edit Distance 从零实现**：Levenshtein 动态规划，WER 的数学基础。  
> 本课将探讨 **CTC 对齐原理**。

## 本课剧情：模型每秒输出 100 个字符，但句子只有 5 个字——这合法吗？

语音识别模型每帧输出一个 token 预测——对于 1 秒音频有约 100 帧。但目标文字可能只有 5 个字符。模型怎么把"100 个预测"压缩成"5 个字符"？

**CTC（Connectionist Temporal Classification）的答案**：引入 blank 符号 `∅`。

想象你在用手机打字，但键盘有延迟，你按了 `a-a-a-∅-b-b-∅-c`，手机输出 `abc`——这就是 CTC。

**CTC 路径 → 目标序列，两步压缩**：
```
路径 π  = [a, a, ∅, b, b, ∅, ∅, c]   （模型帧级输出，含重复和blank）
步骤1   → [a, ∅, b, ∅, ∅, c]           去相邻重复（a-a→a, b-b→b）
步骤2   → [a, b, c]                     去blank
```

**CTC 为什么有效**：一个目标序列 `"abc"` 对应**指数级**的路径数（各种blank位置组合都合法），模型对所有这些路径的概率求和就是输出序列的总概率。训练时最大化这个总概率，不需要预先标注"哪帧对应哪个字符"。

**贪婪解码（greedy decode）**：每帧取概率最大的 token，然后做压缩。快速，但不保证最优：

```
logits (T, V) → argmax每帧 → 路径π → collapse() → 目标序列
```

本节任务：实现 `ctc_greedy_decode(logits, blank=0)`，验证在已知 logits 上正确输出。

In [ ]:
import numpy as np

## 1. CTC 路径与路径概率

设输入序列长度为 `T`，词表（vocabulary）大小为 `V`（含 blank=0）。
模型在每一帧 `t` 输出一个概率分布 `p(token | t)`，形状 `(T, V)`。

**CTC 路径** `π` 是长度为 `T` 的 token 序列（每帧一个符号，含 blank）。
路径概率为各帧概率之积：

```
P(π) = ∏_{t=1}^{T}  p(π_t | t)
```

**CTC 损失** 最大化所有「压缩后等于目标 `y`」的路径的概率之和：

```
P(y | x) = Σ_{π : collapse(π)=y}  P(π)
```

路径数量是指数级的，实际用**前向-后向算法**动态规划（dynamic programming，DP）精确计算。

In [ ]:
# 演示：3帧，词表={blank=0, a=1, b=2}
# 构造一个简单的对数概率矩阵（手动指定）
T, V = 3, 3
log_probs = np.array([
    [np.log(0.1), np.log(0.8), np.log(0.1)],  # 帧0：a 最大
    [np.log(0.7), np.log(0.2), np.log(0.1)],  # 帧1：blank 最大
    [np.log(0.1), np.log(0.1), np.log(0.8)],  # 帧2：b 最大
])
# 列举路径 (a, blank, b) 和 (a, a, b) 两条路径，两者压缩后都是 [a, b]
path1 = [1, 0, 2]  # a ∅ b  -> collapse -> a b
path2 = [1, 1, 2]  # a a b  -> collapse -> a b
p_path1 = np.exp(log_probs[0,1] + log_probs[1,0] + log_probs[2,2])
p_path2 = np.exp(log_probs[0,1] + log_probs[1,1] + log_probs[2,2])
print(f'P(a ∅ b) = {p_path1:.4f}')
print(f'P(a a b) = {p_path2:.4f}')
print(f'P(y=[a,b] | x) ≈ {p_path1 + p_path2:.4f}  (只枚举了两条路径)')


## 2. 压缩规则（collapse）

给定一条路径 `π`（含 blank），压缩为目标序列的规则分两步：

```
步骤1：去掉相邻重复  [a, a, ∅, b, b, ∅, c] → [a, ∅, b, ∅, c]
步骤2：去掉 blank    [a, ∅, b, ∅, c]       → [a, b, c]
```

注意 blank 的作用之一是**分隔同字符**：
路径 `[a, a, b]` 压缩为 `[a, b]`；
路径 `[a, ∅, a, b]` 压缩为 `[a, a, b]`（两个 a 之间有 blank 则不合并）。

In [ ]:
def collapse(path, blank=0):
    """CTC 路径压缩：先去相邻重复，再去 blank。"""
    deduped = [p for i, p in enumerate(path) if i == 0 or p != path[i-1]]
    return [p for p in deduped if p != blank]

# 示例验证
examples = [
    ([1, 1, 0, 2, 2, 0, 3], 'a a ∅ b b ∅ c'),
    ([1, 0, 1, 2],           'a ∅ a b'),
    ([1, 1, 2],              'a a b'),
    ([0, 0, 1, 0],           '∅ ∅ a ∅'),
]
for path, desc in examples:
    result = collapse(path)
    print(f'  [{desc}]  →  {result}')


## 3. 贪婪解码（Greedy Decode）

精确 CTC 解码需要 beam search（枚举所有路径太慢）。
**贪婪解码**是最快的近似：每帧直接取 argmax，得到一条路径，再执行 collapse。

```
π_greedy = [argmax p(token | t)  for t in 1..T]
y_greedy = collapse(π_greedy)
```

贪婪解码不是最优的——它只考虑了最高概率的单条路径，而正确答案可能由多条次优路径联合撑起来。
但在 Whisper、DeepSpeech 等实践中，它作为快速 baseline 效果已经不错。

In [ ]:
# 演示贪婪解码
np.random.seed(42)
T_demo, V_demo = 7, 5  # 7帧，词表大小5（0=blank, 1=a, 2=b, 3=c, 4=d）
logits_demo = np.random.randn(T_demo, V_demo)
# 手动拉高某些帧让结果更直观
logits_demo[0, 1] = 5   # 帧0: a
logits_demo[1, 0] = 5   # 帧1: blank
logits_demo[2, 0] = 5   # 帧2: blank
logits_demo[3, 2] = 5   # 帧3: b
logits_demo[4, 2] = 5   # 帧4: b
logits_demo[5, 0] = 5   # 帧5: blank
logits_demo[6, 3] = 5   # 帧6: c

preds = np.argmax(logits_demo, axis=1)
vocab = {0:'∅', 1:'a', 2:'b', 3:'c', 4:'d'}
print('帧级 argmax  :', [vocab[p] for p in preds])
print('去相邻重复  :', [vocab[p] for i,p in enumerate(preds) if i==0 or p!=preds[i-1]])
decoded = collapse(preds.tolist())
print('最终解码结果:', [vocab[p] for p in decoded], '→ token ids:', decoded)


## 4. ✏️ 实现 `ctc_greedy_decode(logits, blank=0)`

**签名**：
```python
def ctc_greedy_decode(logits: np.ndarray, blank: int = 0) -> list:
    # logits: (T, V) numpy 数组（未经 softmax 的原始得分）
    # 返回: list[int]，压缩后的 token id 序列（不含 blank）
```

**3步实现**：

| 步骤 | 操作 | 工具 |
|---|---|---|
| 1 | `path = logits.argmax(axis=1)` | 每帧取最大 token |
| 2 | 去相邻重复：`[a,a,b,b] → [a,b]` | 循环或 `itertools.groupby` |
| 3 | 去 blank：过滤掉 blank token | list comprehension |

**验收标准**：
- 全 blank logits → `[]`（空列表）
- `[a, a, ∅, b, ∅]` → `[a, b]`
- `[a, ∅, a]` → `[a, a]`（两个 a 中间有 blank，不合并）

In [ ]:
def ctc_greedy_decode(logits: np.ndarray, blank: int = 0) -> list:
    """CTC 贪婪解码：argmax 每帧，去相邻重复，去 blank。

    Args:
        logits: shape (T, V) numpy 数组（未经 softmax 的原始得分）
        blank:  blank 符号的 token id，默认 0
    Returns:
        解码后的 token id 列表
    """
    # ✏️ TODO: 第1步——每帧取 argmax
    preds = None

    # ✏️ TODO: 第2步——去相邻重复（保留第一个，后续只保留与前一个不同的）
    deduped = None

    # ✏️ TODO: 第3步——去掉 blank，返回 list[int]
    raise NotImplementedError("请先完成上方三个 TODO 步骤，再运行检验 cell；卡住可看 solutions/L68_ctc_alignment_solutions.md")

In [ ]:
# --- 检查 ctc_greedy_decode ---

V_check = 5
logits_check = np.full((7, V_check), -10.0)
forced_ids = [1, 0, 0, 2, 2, 0, 3]
for t, tok in enumerate(forced_ids):
    logits_check[t, tok] = 10.0

try:
    result = ctc_greedy_decode(logits_check, blank=0)
    assert result == [1, 2, 3], f'期望 [1,2,3]，实际得到 {result}'
    print('✅ ctc_greedy_decode([a ∅ ∅ b b ∅ c]) =', result, '= [a, b, c]')

    # 边界：全 blank 输入 → 空列表
    logits_blank = np.zeros((5, V_check))
    logits_blank[:, 0] = 10.0
    assert ctc_greedy_decode(logits_blank) == [], '全 blank 应返回空列表'
    print('✅ 全 blank 输入 → []')

    # 边界：单帧单字符
    logits_single = np.array([[-1.0, 5.0, -1.0]])
    assert ctc_greedy_decode(logits_single) == [1]
    print('✅ 单帧 argmax=1 → [1]')

except NotImplementedError:
    print('⬜ 请先实现 ctc_greedy_decode 的三个 TODO 步骤，再运行此检验 cell')


## 5. 参数实验：随机 logits 下的序列压缩率

**实验设置**：
- `T = 50`（输入帧数），`vocab_size = 30`（含 blank=0）
- 跑 1000 次，每次随机采样 `logits ~ N(0,1)`
- 统计 `len(decoded) / T` 的均值和分布

**预期现象**：
- 每帧随机时，blank 被选中概率约 `1/30 ≈ 3.3%`，相邻重复概率约 `1/30`
- 去重+去blank 后，平均输出长度约为输入的 `~93-94%`
  - 推导：每帧选到与前一帧相同 token 的概率约 1/V，去重后存活率约 (V-1)/V；
    再去 blank（选中概率 1/V），两步独立近似后，
    期望压缩率 ≈ ((V-1)/V)² = (29/30)² ≈ 0.934
- 实际语音模型训练后 blank 被大量预测，压缩率会**远低于**此随机基线（blank 多则输出更短）

In [ ]:
try:
    np.random.seed(0)
    T_exp, V_exp, N = 50, 30, 1000

    ratios = []
    for _ in range(N):
        logits_exp = np.random.randn(T_exp, V_exp)
        decoded_exp = ctc_greedy_decode(logits_exp, blank=0)
        ratios.append(len(decoded_exp) / T_exp)

    ratios = np.array(ratios)
    print(f'输入长度 T = {T_exp}，词表大小 V = {V_exp}（含blank）')
    print(f'1000次随机实验：')
    print(f'  平均输出长度     : {ratios.mean()*T_exp:.1f} 帧')
    print(f'  平均压缩率 len/T : {ratios.mean():.3f}')
    print(f'  最小/最大压缩率  : {ratios.min():.3f} / {ratios.max():.3f}')
    print()
    expected = ((V_exp - 1) / V_exp) ** 2
    print(f'期望压缩率公式 ((V-1)/V)² = ({V_exp-1}/{V_exp})² ≈ {expected:.3f}')
    print('（双步独立近似：先去相邻重复，再去 blank；随机 logits 下两步同量级）')

except NotImplementedError:
    print('⬜ 请先实现 ctc_greedy_decode，再运行参数实验')


## 6. 补充：CTC 的单调路径约束

标题中提到的 **单调路径**（monotonic paths）是 CTC 的一个关键结构约束，
在贪婪解码中隐含成立，在动态规划（L69）中需要显式建模。

**约束定义**：CTC 路径 π 是从帧序列到标签序列的映射，满足：
- 每一帧只能"停留在当前标签/blank"或"推进到下一个标签"，不能回退。
- 形式上：路径中标签的出现顺序严格非递减——帧轴单调向右，标签轴不倒退。

**直觉**：语音是时序信号，字符的声学实现只能从左到右出现，
不可能先说完 "o" 再回头说 "h"（hol 的 h 必须在 o 之前）。
CTC 用单调约束把这个物理事实内嵌到路径结构中。

**与贪婪解码的关系（本节）**：贪婪解码每帧独立取 argmax，
再经 collapse 得到输出。单调性是**自动满足**的：
帧的处理顺序本身就是从左到右，collapse 只做过滤，不会打乱顺序。

**与动态规划的关系（L69 预告）**：精确 CTC 前向算法需要对所有满足
`collapse(π) = y` 的路径求概率之和。
单调约束将路径空间从 V^T 压缩为只考虑两种转移——
"在当前 blank/标签上延伸" 或 "推进到下一个标签"——
从而可以用 O(T × |y|) 的动态规划高效精确计算。


## 附录 C · 从贪婪解码到前向算法：L69 预习（8 分钟）

1. **L68 做了什么**：找**一条**最优路径（贪婪近似）。
2. **L69 要做什么**：对**所有**合法路径的概率**求和**（精确训练损失）。
3. **为何需要 DP**：路径数 O(V^T)；DP 降到 O(T·S)，其中 `S = 2L + 1`。
4. **与 L67 的关系**：编辑距离 DP 也是“填表递推”；CTC 只是把表里的值换成 log 概率。
5. **log 域一句**：概率相乘 → log 域用 `logaddexp` 做“log 下的加法”。

```text
        s=0   s=1   s=2   …   (扩展标签轴)
t=0     α     ·     ·
t=1     ·     ·     ·
…
```

In [ ]:
from itertools import product

toy_labels = [1, 2]  # a b
toy_paths = [
    path
    for path in product(range(3), repeat=3)
    if collapse(path, blank=0) == toy_labels
]
print("T=3 时的合法路径数：", len(toy_paths))
print("前 5 条路径：", toy_paths[:5])
print("精确求和要等 L69 的 ctc_forward；L68 这里只先看路径空间怎么长出来。")

## 本课收束

本节实现了 `ctc_greedy_decode`，它输出一个 `list[int]`（压缩后的 token id 序列），对应 CTC 路径的近似最优解。
该函数直接服务于 `aurora.speech` 推理管线——在 `torch.nn.CTCLoss` 训练完成后，贪婪解码是最快的线上解码方式。
本课附录 C 已把 L69 的前向算法预热：先看路径空间，再看 log 域递推，就不会在下一节突然掉进 cliff。
下一节（L69）将实现 CTC 前向算法的完整动态规划，追踪 blank-skip 和标签折叠两条合法路径，计算序列概率。
L69 之后还将讨论 Whisper 选择交叉熵而非 CTC 的原因。

## ✏️ 白板挑战：CTC 解码手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：路径 `[a, ∅, a, a, ∅, b]` 压缩成什么目标序列？  
（去相邻重复: [a,∅,a,∅,b]，去blank: [a,a,b]）

**问 2**：目标序列 `"ab"` 对应哪些长度=4的 CTC 路径？（blank=0, a=1, b=2）  
（需要先有a序列再b序列，中间可插blank；如[a,a,∅,b],[a,∅,b,b],[∅,a,b,b],[a,∅,∅,b]等）

**问 3**：贪婪解码与最优解码的区别是什么？什么情况下贪婪解码会出错？  
（贪婪=每帧argmax；当最优路径需要"当前帧选次优token，下帧选最优"时贪婪失效，beam search更好）

**问 4**：CTC 为什么不需要帧级标注（forced alignment）？  
（总概率=所有合法路径的概率和；只需序列级标注"这段音频对应这句话"即可训练）

**问 5**：若 logits shape=(50, 30)，贪婪解码后最长可能输出多少个 token？  
（最多50个，若每帧都是不同非blank token；最少0个，若全是blank或全是同一token）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：路径压缩
path_q1 = [1, 0, 1, 1, 0, 2]  # a=1,blank=0,b=2
result_q1 = collapse(path_q1, blank=0)
assert result_q1 == [1, 1, 2], f"得到{result_q1}，期望[1,1,2]"
print(f"Q1 ✅  [a,∅,a,a,∅,b]压缩后={result_q1} = [a,a,b]（两个a中间有blank，不合并）")

# 问2：长度=4路径枚举（验证collapse输出正确）
target_q2 = [1, 2]  # ab
valid_paths = []
for p0 in range(3):
    for p1 in range(3):
        for p2 in range(3):
            for p3 in range(3):
                path = [p0,p1,p2,p3]
                if collapse(path, blank=0) == target_q2:
                    valid_paths.append(path)
assert len(valid_paths) >= 4, f"找到{len(valid_paths)}条路径，应≥4"
print(f"Q2 ✅  目标'ab'对应{len(valid_paths)}条长度=4的CTC路径（如{valid_paths[:3]}...）")

# 问3：贪婪 vs 最优（演示场景）
try:
    # 构造一个贪婪次优的logits
    logits_q3 = np.zeros((3, 3))
    logits_q3[0, 0] = 1.0   # 第0帧：blank最优
    logits_q3[1, 1] = 0.9   # 第1帧：token1次优
    logits_q3[1, 0] = 1.0   # 第1帧：blank最优
    logits_q3[2, 2] = 1.0   # 第2帧：token2最优
    greedy_path = logits_q3.argmax(axis=1)
    greedy_result = collapse(greedy_path.tolist(), blank=0)
    print(f"Q3 ✅  贪婪解码路径={greedy_path.tolist()} → {greedy_result}")
    print(f"       beam search可能找到更好路径（此例中贪婪=最优，复杂情况下不同）")
except Exception as e:
    print(f"Q3 ✅  贪婪=每帧argmax；beam search探索多路径，更准确但更慢")

# 问4：序列级训练（概念验证 — 使用ctc_greedy_decode）
try:
    logits_q4 = np.array([[0.1, 5.0, 0.1],  # 帧0: token1最大
                           [0.1, 0.1, 5.0],  # 帧1: token2最大
                           [5.0, 0.1, 0.1]]) # 帧2: blank最大
    decoded = ctc_greedy_decode(logits_q4, blank=0)
    assert decoded == [1, 2], f"解码结果={decoded}"
    print(f"Q4 ✅  ctc_greedy_decode: logits→{decoded}（只需序列标注[1,2]，无需帧对齐）")
except NotImplementedError:
    print(f"Q4 ✅  CTC不需帧级标注：优化所有合法路径的概率总和（待实现后验证）")

# 问5：最长/最短输出
try:
    T_q5, V_q5 = 10, 5
    # 最长：每帧交替不同非blank token
    logits_max = np.zeros((T_q5, V_q5))
    for t in range(T_q5): logits_max[t, (t % (V_q5-1)) + 1] = 1.0  # 非blank交替
    decoded_max = ctc_greedy_decode(logits_max, blank=0)
    # 最短：全blank
    logits_min = np.zeros((T_q5, V_q5)); logits_min[:, 0] = 1.0
    decoded_min = ctc_greedy_decode(logits_min, blank=0)
    print(f"Q5 ✅  T=10时: 最多输出{len(decoded_max)}个token，最少{len(decoded_min)}个（全blank）")
except NotImplementedError:
    print(f"Q5 ✅  T帧最多T个token（无相邻重复），最少0个（全blank）（待实现后验证）")
print("\n🎉 CTC解码白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l68_review = {
    "ctc_blank_understood":      None,  # 理解blank符号的作用：允许帧数>>字符数，支持字符重复？True/False
    "collapse_rules":            None,  # 记住压缩2步：去相邻重复→去blank？True/False
    "greedy_decode_impl":        None,  # ctc_greedy_decode()实现正确，3个测试用例通过？True/False
    "no_alignment_needed":       None,  # 理解CTC为什么不需要帧级对齐标注？True/False
    "whiteboard_passed":         None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l68_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l68_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L68 全部通关！进入 L69：CTC 前向算法')

---

→ **下一课**　[L69 · CTC 前向算法](L69_ctc_forward.ipynb)

> 下节课将学习 **CTC 前向算法**：所有路径概率求和，O(T·S) 纯 NumPy 实现。先读本课附录 C，再回到 L67 的 DP 填表逻辑。